# Train classification model

Imports

In [ ]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F

# Build an absolute path from this notebook's parent directory
src_module_path = os.path.abspath(os.path.join('..', 'src'))

# Add to sys.path if not already present
if src_module_path not in sys.path:
    sys.path.append(src_module_path)

from data import get_data_loaders, show_image
from models.classification_model import ClassificationModel
import config

Data Initialization

In [ ]:
trainloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([2, 1, 150, 150])
[tensor([[[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -0.9765, -0.9686,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -0.9686, -0.9686,  ..., -1.0000, -1.0000, -1.0000]]],


        [[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.000

Parameter optimization

In [3]:
model = ClassificationModel()

print_num = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)
for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')



Epoch 1
[1,    10] loss: 2.110
[1,    20] loss: 1.012
[1,    30] loss: 0.839
[1,    40] loss: 0.714
[1,    50] loss: 0.777
[1,    60] loss: 0.693
[1,    70] loss: 0.705
[1,    80] loss: 0.764
[1,    90] loss: 0.714
[1,   100] loss: 0.707
[1,   110] loss: 0.738
[1,   120] loss: 0.737
[1,   130] loss: 0.711
[1,   140] loss: 0.686
[1,   150] loss: 0.725
[1,   160] loss: 0.956
[1,   170] loss: 0.675
[1,   180] loss: 0.735
[1,   190] loss: 0.635
[1,   200] loss: 0.752
[1,   210] loss: 0.645
[1,   220] loss: 0.717
[1,   230] loss: 0.700
[1,   240] loss: 0.613
[1,   250] loss: 0.767
[1,   260] loss: 0.701
[1,   270] loss: 0.709
[1,   280] loss: 0.701
[1,   290] loss: 0.708
[1,   300] loss: 0.719
[1,   310] loss: 0.619
[1,   320] loss: 0.699
[1,   330] loss: 0.734
[1,   340] loss: 0.710
Epoch 2
[2,    10] loss: 0.696
[2,    20] loss: 0.628
[2,    30] loss: 0.734
[2,    40] loss: 0.703
[2,    50] loss: 0.691
[2,    60] loss: 0.706
[2,    70] loss: 0.665
[2,    80] loss: 0.716
[2,    90] loss: 0

In [4]:
dataiter = iter(testloader)
images, labels = next(dataiter)

print(labels)

output = model(images)
print(output)
_, predicted = torch.max(output, 1)
print(predicted)


tensor([0, 1])
tensor([[ -3.9497,   6.6242, -19.3309, -18.0640, -17.3698, -18.8430, -17.2409,
         -18.3621, -16.6524, -17.0562],
        [ -3.8366,   6.5665, -20.9298, -19.5864, -18.7128, -20.8061, -18.8759,
         -20.0244, -18.3790, -18.5276]], grad_fn=<AddmmBackward0>)
tensor([1, 1])


In [5]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 75 %


Initialize model

Train

Test